In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic, shapelets
from gigalens.jax.profiles.mass import epl, shear

import jax
from jax import random
import numpy as np
import json
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
import blackjax
import importlib
import os
tfd = tfp.distributions

In [ ]:
from gigalens_research.inference import MCLMC
from gigalens_research.inference_utils import *
from gigalens_research.plotting import *
from gigalens_research.simulations import load_vela_source
from gigalens_research.simtests.pipelines import build_map_bootstrap_mclmc, PartialTruthBootstrapQzStage
from gigalens_research.simtests.system import from_vela_dir
# from gigalens_research.simtests.experiments import vela_shapelets
from gigalens_research.simtests.experiments import vela_elliptical_sersiclets
from gigalens_research.simtests.registry import get_inference_builder

In [ ]:
home = os.path.expanduser("~/")
srcdir = os.path.join(home, "gigalens/src/")
data_dir = os.path.join(home, f"GIGALens-Code/data/")
results_dir = os.path.join(home, f"GIGALens-Code/results/test_new_api_elliptical_sersiclets/")

def load_vela_sim_system(sim_num, cam, rep):
    source_plane_dir = os.path.join(data_dir, f"vela_sources/vela{sim_num}_cam{cam}_a0.500_f814w/")
    sim_system_dir = os.path.join(data_dir, f"vela_sim_systems/vela{sim_num}_cam{cam}_rep{str(rep).zfill(2)}_a0.500_f814w/")

    
    psf = np.load(os.path.join(source_plane_dir, "psf.npy"))
    with open(os.path.join(source_plane_dir, "metadata.json")) as f:
        meta = json.load(f)
    
    
    source_img_pixel_scale = meta['source_pixel_scale_arcsec']
    delta_pix = meta['instrument_pixel_scale_arcsec']
    num_pix = 200
    
    sim_config = SimulatorConfig(delta_pix=delta_pix, num_pix=num_pix, supersample=1, kernel=psf)
    
    observed_img = jnp.load(os.path.join(sim_system_dir, "lens_img.npy"))
    
    with open(os.path.join(sim_system_dir, "true_params"), 'rb') as file_handle:
        true_params = pickle.load(file_handle)


    source_img_nJy = np.load(os.path.join(source_plane_dir, "source_image.npy"))
    SB_nJy_per_arcsec2 = source_img_nJy / (source_img_pixel_scale**2)
    SB_cps_per_arcsec2 = SB_nJy_per_arcsec2* 1e-9 /meta["photfnu_Jy"]
    source_image = SB_cps_per_arcsec2

    
    vela_source = load_vela_source(source_plane_dir)
    
    
    return observed_img, true_params, sim_config, vela_source

In [ ]:
# def vela_system_model(sim_config, use_shapelets=True, n_max=10):
#     lens_prior = tfd.JointDistributionSequential(
#         [
#             tfd.JointDistributionNamed(
#                 dict(
#                     theta_E=tfd.LogNormal(jnp.log(1.25), 0.4),
#                     gamma= tfd.TruncatedNormal(2, 0.5, 1, 3), #! CHANGE
#                     e1=tfd.TruncatedNormal(0, 0.2, -0.5, 0.5),
#                     e2=tfd.TruncatedNormal(0, 0.2, -0.5, 0.5),
#                     center_x=tfd.Normal(0, 0.06),
#                     center_y=tfd.Normal(0, 0.06),
#                 )
#             ),
#             tfd.JointDistributionNamed(
#                 dict(gamma1=tfd.TruncatedNormal(0, 0.1, -0.5, 0.5), gamma2=tfd.Normal(0, 0.1, -0.5, 0.5))
#             ),
#         ]
#     )
#     lens_light_prior = tfd.JointDistributionSequential(
#         [
#             tfd.JointDistributionNamed(
#                 dict(
#                     R_sersic=tfd.LogNormal(jnp.log(1.6), 0.25),
#                     n_sersic=tfd.Uniform(0.5, 8),
#                     e1=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
#                     e2=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
#                     center_x=tfd.Normal(0, 0.02),
#                     center_y=tfd.Normal(0, 0.02),
#                     # Ie=tfd.LogNormal(jnp.log(300.0), 0.5),
#                 )
#             )
#         ]
#     )
    
    
#     # amp_prior = {key: tfd.Normal(0,500/float(jnp.sqrt(i+1))) for i, key in enumerate(shapelets.Shapelets(n_max)._amp_names)}
    

#     if use_shapelets:
#         source_light_prior = tfd.JointDistributionSequential(
#             [
#                 tfd.JointDistributionNamed(
#                     dict(
#                         beta=tfd.LogNormal(jnp.log(0.7), 0.4),
#                         center_x=tfd.Normal(0, 0.5),
#                         center_y=tfd.Normal(0, 0.5),
#                         # **amp_prior
#                     )
#                 ),
#             ]
#         )
#     else:
#         lens_light_prior = tfd.JointDistributionSequential(
#             [
#                 tfd.JointDistributionNamed(
#                     dict(
#                         R_sersic=tfd.LogNormal(jnp.log(1.6), 0.4),
#                         n_sersic=tfd.Uniform(0.5, 8),
#                         e1=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
#                         e2=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
#                         center_x=tfd.Normal(0, 0.5),
#                         center_y=tfd.Normal(0, 0.5),
#                     )
#                 )
#             ]
#         )
    
    
#     prior = tfd.JointDistributionSequential(
#         [lens_prior, lens_light_prior, source_light_prior]
#     )
    
#     # n_max = 10
#     src_model = shapelets.ShapeletsFast(n_max=n_max, use_lstsq=True, interpolate=False) if shapelets else sersic.SersicEllipse(use_lstsq=True)
#     phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=True)], [src_model])
#     lens_sim = LensSimulator(phys_model, sim_config, bs=1)
        
#     prob_model = BackwardProbModel(prior, observed_img, background_rms=background_rms, exp_time=exp_time)
#     model_seq = ModellingSequence(phys_model, prob_model, sim_config)

#     return model_seq, lens_sim

In [ ]:
cam = "12"
sim_num = "01"
rep = 3
n_max=25

# background_rms = 0.002
# exp_time = 2000

print(f"--------------------------- {sim_num}-{str(rep).zfill(2)} ---------------------------------")
system_name = f"vela{sim_num}_cam{cam}_rep{str(rep).zfill(2)}_a0.500_f814w"
system_id = f"vela{sim_num}_cam{cam}_rep{str(rep).zfill(2)}"
source_name = f"vela{sim_num}_cam{cam}_a0.500_f814w"
# sim_system_dir = os.path.join(data_dir, f"vela_sim_systems/{system_name}/")
# save_dir = os.path.join(results_dir, f"vela{sim_num}_cam{cam}_rep{str(rep).zfill(2)}_a0.500_f814w", f"n_max{str(n_max).zfill(2)}/")
# os.makedirs(save_dir, exist_ok=True)

system = from_vela_dir(
    system_dir=f"/global/homes/l/linusu/GIGALens-Code/data/vela_sim_systems/{system_name}",
    source_dir=f"/global/homes/l/linusu/GIGALens-Code/data/vela_sources/{source_name}",
    system_id="vela01_cam12_rep00",
    delta_pix=0.03,   # read automatically if metadata.json is present
    num_pix=200,
    supersample=1,
    background_rms=0.002,
    exp_time=2000.0,
)
system.likelihood_precision = "float64"
system.conv_precision = "float32"

# system.high_precision_likelihood = True
#* Get truth to start it (to make MAP less intensive)
# model_seq_fixed_lens, lens_sim_fixed_lens = free_source_fixed_lens_model(sim_config, true_params, use_shapelets=True, n_max=n_max)
# model_seq, lens_sim = vela_system_model(sim_config, use_shapelets=True, n_max=n_max)


model_seq = get_inference_builder("epl_shear_sersic_elliptical_sersiclets")(system, n_max=n_max)




In [ ]:
observed_img, true_params, sim_config, vela_source = load_vela_sim_system(sim_num, cam, rep)  

In [ ]:
true_params

In [ ]:
pipeline = Pipeline(InferenceContext.from_modelling_sequence(model_seq))
# pipeline.add(MAPStage(num_steps=1000, n_samples=200))
# pipeline.add(SVIStage(num_steps=1000, n_vi=200))

# bridge = BridgeStage(
#     name="diag_qz_from_map",
#     version="v2",
#     requires=("z_best",),
#     produces=("qz",),
#     fn=lambda z_best: tfd.MultivariateNormalDiag(
#         loc=jnp.asarray(z_best),
#         scale_diag=jnp.full(z_best.shape[-1], 1e-8),
#     ),
# )
# pipeline.add(bridge)

# pipeline.add(HessianSurrogateStage())

pipeline.add(
    PartialTruthBootstrapQzStage(
        system=system,
        free=("source",),
        map_num_steps=200,
        map_n_samples=50,
    ))

pipeline.add(MCLMCStage(n_chains=8, num_burnin_steps=2000, num_results=2000, progress_bar=True, debug=True))

In [ ]:
artifacts = pipeline.run(resume=False, out_dir=os.path.join(results_dir, f"{system_name}"))

In [ ]:
os.path.join(results_dir, f"{system_name}")

In [ ]:

from gigalens_research.inference_utils import truth_source_from_light_model

In [ ]:
ps = pipeline.posterior()
observed_img, true_params, sim_config, vela_source = load_vela_sim_system(sim_num, cam, rep)  

truth_fn = truth_source_from_light_model(vela_source.light, true_params[2][0])
report = PosteriorReport(ps, truth_x=true_params, truth_source_fn=truth_fn)
pipeline_report = PipelineReport(pipeline)

In [ ]:
report.source_panel()
plt.show()

In [ ]:

fig = pipeline_report.diagnostics("mclmc", chain=3)
fig.show()

In [ ]:
report.full_report()

In [ ]:

# fig = pipeline_report.compound_corner()

In [ ]:
fig = pipeline_report.image_comparison()

In [ ]:
fig = pipeline_report.loss_histories()
